# Korea Food-Service Permits: 3M-Row Quickstart

This notebook is a lightweight starting point for the **Korea Food-Service Permits - Current Snapshot** dataset.

The dataset contains **3,010,802 current permit records** for general restaurants, rest cafes, and bakeries across South Korea. The same 26-column table is available as CSV and Parquet; this notebook uses Parquet for fast selective reads.

This is a **current snapshot**, not a historical time series or lossless lifecycle event log.


In [ ]:
from pathlib import Path
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pandas as pd
import matplotlib.pyplot as plt

input_root = Path('/kaggle/input')
parquet_matches = sorted(input_root.rglob('korea_food_service_permits.parquet'))
csv_matches = sorted(input_root.rglob('korea_food_service_permits.csv'))
if not parquet_matches or not csv_matches:
    visible = [str(path.relative_to(input_root)) for path in list(input_root.rglob('*'))[:80]]
    raise FileNotFoundError(f'Dataset input files not attached. Visible inputs: {visible}')
PARQUET = parquet_matches[0]
CSV = csv_matches[0]
DATA_DIR = PARQUET.parent

print('Files:')
for path in sorted(DATA_DIR.iterdir()):
    print(f'  {path.name:40s} {path.stat().st_size / 1024**2:9.1f} MiB')


## 1. Verify the published shape without loading 3 million rows into pandas

In [ ]:
pf = pq.ParquetFile(PARQUET)
print(f'Rows:    {pf.metadata.num_rows:,}')
print(f'Columns: {pf.metadata.num_columns}')
print(f'Row groups: {pf.metadata.num_row_groups}')
print()
print('First 10 column names:')
print(pf.schema_arrow.names[:10])


## 2. Category coverage

In [ ]:
source_table = pq.read_table(PARQUET, columns=['source_key'])
source_counts = source_table.group_by('source_key').aggregate([('source_key', 'count')]).to_pandas()
source_counts = source_counts.rename(columns={'source_key_count': 'records'}).sort_values('records', ascending=False)
source_counts


In [ ]:
ax = source_counts.set_index('source_key')['records'].plot(kind='bar', figsize=(8, 4), title='Records by source category')
ax.set_ylabel('records')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


## 3. Raw source status distribution

In [ ]:
status_table = pq.read_table(PARQUET, columns=['source_key', 'source_status_code', 'source_status_name'])
status_counts = (
    status_table
    .group_by(['source_key', 'source_status_code', 'source_status_name'])
    .aggregate([('source_key', 'count')])
    .to_pandas()
    .rename(columns={'source_key_count': 'records'})
    .sort_values(['source_key', 'records'], ascending=[True, False])
)
status_counts.head(20)


**Interpretation caution:** source status codes/names are preserved as reported. This project does not force them into a canonical active/closed/terminal label. In observed evidence, status `03` is not irreversible, and status `05` remains semantically unresolved.


## 4. Permit-year coverage

In [ ]:
permit_dates = pq.read_table(PARQUET, columns=['permit_date'])['permit_date']
years = pc.year(permit_dates)
year_table = pa.table({'permit_year': years})
year_counts = (
    year_table
    .group_by('permit_year')
    .aggregate([('permit_year', 'count')])
    .to_pandas()
    .rename(columns={'permit_year_count': 'records'})
    .dropna()
    .sort_values('permit_year')
)
year_counts.tail(20)


`permit_date` is an administrative permit date. It should **not** automatically be interpreted as the physical opening date of a business.


## 5. Human-readable sample

In [ ]:
sample_columns = [
    'source_key', 'authority_code', 'management_number', 'permit_date',
    'source_status_name', 'source_detail_status_name', 'closure_date',
    'business_name', 'business_type_name', 'road_address',
    'source_coordinate_x', 'source_coordinate_y',
]
sample = pf.read_row_group(0, columns=sample_columns).slice(0, 10).to_pandas()
sample


## 6. CSV users

The CSV contains the **same 3,010,802 rows and 26 columns** as the Parquet file. It is about 1.4 GB, so for exploratory work you may want to read only selected columns or use chunked loading.


In [ ]:
# Small CSV preview without loading the full 1.4 GB file
csv_preview = pd.read_csv(CSV, nrows=5)
csv_preview


## 7. Important semantic and geospatial notes

- `management_number` is retained as a **bounded continuity candidate**, not asserted as an official primary key.
- `permit_date` is not necessarily the physical opening date.
- `closure_date` is contextual and is not assumed to be a permanent terminal event.
- `source_coordinate_x` / `source_coordinate_y` use **EPSG:5174** for this verified snapshot (X=easting, Y=northing).
- The separately derived WGS84 sidecar is **not included** in this public dataset.
- Partial history is **not included**; this release should not be described as a time series.

See `DATA_DICTIONARY.md`, `schema.json`, `SOURCES.md`, and `release-manifest.json` in the dataset for exact documentation and provenance.
